In [0]:
%python
df = spark.read.table('project_src_data.students_raw.student_records')
df.display()

In [0]:
%python
from pyspark.sql.functions import count,col

group_df = df.groupBy('schema_tag').agg(count('*'))

In [0]:
%python
display(group_df)

In [0]:
%python
gender_df = df.filter(col('gender').isNotNull())
display(gender_df)

In [0]:
%python
filtered_df = df.filter(~col('phone').like("+91%"))
display(filtered_df)

In [0]:
%python
df_new = df.select('student_id', 'full_name',
                   'email', 'phone', 'city',
                   'department', 'cgpa', 'sports', 
                   'placement_status'
                   )

In [0]:
%python
df_new.display()

In [0]:
%python
from pyspark.sql.functions import col
df_fil = df_new.filter(col("placement_status").isNull())

In [0]:
%python
df_fil.display()

In [0]:
%python
from pyspark.sql.functions import col,when

df_upd = df_new.withColumn(
    "placement_status",
    when(col("placement_status").isNull(), "Not Placed").otherwise(col("placement_status"))
)

In [0]:
%python
display(df_upd)

In [0]:
%python
from pyspark.sql.functions import col,when

df_upd = df_upd.withColumn(
    "placement_status",
    when(col("placement_status") == "UNKNOWN", "Not Placed").otherwise(col("placement_status"))
)

In [0]:
%python
df_upd.display()

In [0]:
%python
df_n = df_upd.filter(col("placement_status") == "#VALUE!")

In [0]:
%python
df_n.display()

In [0]:
%python
df_upd = df_upd.withColumn(
    "placement_status",
    when(col("placement_status") == "#VALUE!", "Not Placed").otherwise(col("placement_status"))
)
df_upd.display()

In [0]:
%python
valid_status = ["In Process", "Opted Out", "Not Placed", "Placed", "Appeared"]

df_fil = df_upd.filter(~col("placement_status").isin(valid_status))

display(df_fil)

In [0]:
%python
valid_status = ["In Process", "Opted Out", "Not Placed", "Placed", "Appeared"]

df_upd = df_upd.withColumn(
    "placement_status",
    when(
        col("placement_status").isin(valid_status),
        col("placement_status")
    ).otherwise("Not Placed")
)
df_upd.display()

In [0]:
%python
df_fil = df_upd.filter(col("sports").isNull())
df_fil.display()

In [0]:
%python
df_upd = df_upd.withColumn(
    "sports",
    when(
        col("sports").isNull(), "Cricket"
    ).otherwise(col("sports")
))

In [0]:
%python
df_upd.display()

In [0]:
%python
non_valid = ["TBD", "#VALUE!", "null", "None", "N/A", "NULL", "na", "???", 'n/a', '-']

df_upd = df_upd.withColumn(
    "sports",
    when(
        col("sports").isin(non_valid),
        "Not Interested"
    ).otherwise(col("sports")
    )
)

In [0]:
%python
df_upd.display()

In [0]:
%python
from pyspark.sql.functions import col, avg

avg_value = df_upd.filter(
    (col("cgpa") >= 5.5) & (col("cgpa") <= 10)
).select(avg("cgpa")).first()[0]
print(avg_value)

In [0]:
%python
df_upd = df_upd.withColumn(
    "cgpa",
    when(
        (col("cgpa").isNull()) |
        (col("cgpa") < 5.5) |
        (col("cgpa") > 10),
        avg_value
    ).otherwise(col("cgpa"))
)
df_upd.display()

In [0]:
%python
from pyspark.sql.functions import col, round

df_upd = df_upd.withColumn(
    "cgpa",
    round(col("cgpa"), 2)
)
df_upd.display()

In [0]:
%python
from pyspark.sql.functions import count,col

dff = df_upd.groupBy(col('department')).count()
dff.display()

In [0]:
%python

not_valid = ["NULL", "N/A", "TBD", "na", "UNKNOWN", "#VALUE!", "???", "null", "n/a"]

df_upd =df_upd.withColumn(
    "department",
    when(
        col("department").isin(not_valid),
        "Unknown"
    ).otherwise(col("department")
    )
)
df_upd.display()

In [0]:
%python
dff = df_upd.groupBy(col('department')).count()
dff.display()

In [0]:
%python
df_upd.display()

In [0]:
%python
dff= df_upd.groupBy(col("city")).count()
dff.display()

In [0]:
%python
abroad_cities = ["London", "New York", "Toronto", "Singapore", "Sydney"]
not_valid = ["null", "n/a", "-", "na", "TBD", "#VALUE!"]

df_upd = df_upd.withColumn(
    "city",
    when(
        col("city").isin(abroad_cities),
        "Abroad"
    ).when(
        col("city").isin(not_valid),
        "Pune"
    ).when(
        col("city") == "Bangalore",
        "Bengaluru"
    ).otherwise(col("city")))

df_upd.display()

In [0]:
%python
dff = df_upd.groupBy("city").count()
dff.display()

In [0]:
%python
from pyspark.sql.functions import regexp_replace

df_upd = df_upd.withColumn(
    "phone",
    regexp_replace(col("phone"), r"^\+91[- ]?|^0+", "")
)
df_upd.display()

In [0]:
%python
df_upd = df_upd.withColumn(
    "phone",
    regexp_replace(col("phone"), r"^\+91[- ]?|^0+", "")
)

In [0]:
%python
df_upd.display()

In [0]:
%python
dff= df_upd.groupBy("phone").count()
dff.display()

In [0]:
%python
from pyspark.sql.functions import col, when

non_valid = ["9999999999", "XXXXXXXXXX"]

df_upd = df_upd.withColumn(
    "phone",
    when(col("phone").isin(non_valid), None)
    .otherwise(col("phone"))
)
df_upd.display()

In [0]:
%python
from pyspark.sql.functions import col

df_valid = df_upd.filter(
    ~col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
)
df_valid.display()

In [0]:
%python
email_regex = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

df_upd = df_upd.withColumn(
    "email",
    when(
        col("email").rlike(email_regex),
        col("email")
    ).otherwise(None)
)

df_upd.display()

In [0]:
%python
from pyspark.sql.functions import col, initcap, trim, regexp_replace

df_upd = df_upd.withColumn(
    "full_name",
    initcap(                                  # Capitalize each word
        regexp_replace(                       # fix multiple spaces
            trim(col("full_name")),           # remove leading/trailing spaces
            r"\s+", " "                       # replace multiple spaces → one space
        )
    )
)
df_upd.display()

In [0]:
%python
from pyspark.sql.functions import col

df_invalid = df_upd.filter(
    ~col("full_name").rlike(r"^[A-Z][a-z]+ [A-Z][a-z]+$")
)
df_invalid.display()

In [0]:
%python
from pyspark.sql.functions import col, regexp_replace, initcap

df_upd = df_upd.withColumn(
    "full_name",
    initcap(
        regexp_replace(col("full_name"), "_", " ")
    )
)
df_upd.display()

In [0]:
%python
from pyspark.sql.functions import col

df_invalid = df_upd.filter(
    ~col("full_name").rlike(r"^[A-Z][a-z]+ [A-Z][a-z]+$")
)
df_invalid.display()

In [0]:
%python
from pyspark.sql.functions import col, split, initcap, regexp_replace, when


invalid_ls = ["#value!", "Tbd", "-", "Null", "???", "N/a", "Unknown", "Na"]

df_upd = df_upd.withColumn(
    "full_name",
    when(
        col("full_name").isin(invalid_ls) & col("email").rlike(r"^[a-zA-Z]+[._][a-zA-Z]+@"),
        initcap(
            regexp_replace(
                split(col("email"), "@").getItem(0),
                r"[._]", " "
            )
        )
    ).otherwise(col("full_name"))
)

df_upd.display()


In [0]:
%python
from pyspark.sql.functions import col

df_invalid = df_upd.filter(
    ~col("full_name").rlike(r"^[A-Z][a-z]+ [A-Z][a-z]+$")
)
df_invalid.display()

In [0]:
%python
from pyspark.sql.functions import col, split, initcap, regexp_replace, when


invalid_ls = ["#value!", "Tbd", "-", "Null", "???", "N/a", "Unknown", "Na"]

df_upd = df_upd.withColumn(
    "full_name",
    when(
        col("full_name").isin(invalid_ls) & col("email").rlike(r"^[a-zA-Z]+[0-9]*@"),
        # Case 2: manish70 → Manish
        initcap(
            regexp_replace(
                split(col("email"), "@").getItem(0),
                r"[0-9]", ""   # remove numbers
            )
        )
    ).otherwise(col("full_name"))
)

df_upd.display()


In [0]:
%python
from pyspark.sql.functions import col, count

dup_ids = df_upd.groupBy("student_id") \
            .agg(count("*").alias("cnt")) \
            .filter(col("cnt") > 1)

dup_ids.display()

In [0]:
%python
df_upd.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("students_cleaned")

In [0]:
SELECT * FROM students_cleaned;